# 04 — GNN + LSTM Training

Trains the hybrid GNN + LSTM model and ablations:
1. **GNN + LSTM** (full model)
2. **LSTM-only** (no graph structure)
3. **GNN-only** (single window, no temporal context)

Evaluation: LOSO cross-validation on PAMAP2 (and optionally HHAR)

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import PROCESSED_DIR, BATCH_SIZE, SEED
from src.train import set_seed, get_device, loso_splits, train_model
from src.models import GNNLSTMModel, LSTMOnlyModel, GNNOnlyModel
from src.dataset import HARSequenceDataset, HARWindowDataset, HARGraphDataset
from src.graph_construction import build_pamap2_adj

set_seed(SEED)
DEVICE = get_device()
print(f'Device: {DEVICE}')

## 1. Load Data

In [ ]:
try:
    X   = np.load(f'{PROCESSED_DIR}/pamap2_X.npy')
    y   = np.load(f'{PROCESSED_DIR}/pamap2_y.npy')
    subj = np.load(f'{PROCESSED_DIR}/pamap2_subjects.npy')
    print(f'PAMAP2 loaded: X={X.shape}, classes={len(np.unique(y))}')
    N_CLASSES = len(np.unique(y))
    DATA_LOADED = True
except FileNotFoundError:
    print('Run 02_preprocessing_pipeline.ipynb first.')
    DATA_LOADED = False

## 2. Build Datasets

In [ ]:
if DATA_LOADED:
    SEQ_LEN = 10  # number of consecutive windows per LSTM input

    seq_dataset   = HARSequenceDataset(X, y, dataset='pamap2', seq_len=SEQ_LEN)
    flat_dataset  = HARWindowDataset(X, y)
    graph_dataset = HARGraphDataset(X, y, dataset='pamap2')

    x_sample, adj_sample, y_sample = seq_dataset[0]
    print(f'Sequence sample: x={x_sample.shape}, adj={adj_sample.shape}, label={y_sample}')

    N_NODES     = x_sample.shape[1]
    NODE_FEAT   = x_sample.shape[2]
    FLAT_DIM    = flat_dataset[0][0].shape[0]

    print(f'n_nodes={N_NODES}, node_feat_dim={NODE_FEAT}, flat_dim={FLAT_DIM}')

## 3. Quick Single-Fold Training (demo — uses first subject as test)

In [ ]:
if DATA_LOADED:
    from torch.utils.data import DataLoader, Subset

    # Use subject 1 as test, rest as train for this demo
    test_subj = np.unique(subj)[0]
    train_idx = np.where(subj != test_subj)[0]
    test_idx  = np.where(subj == test_subj)[0]

    # Rebuild sequence dataset subjects index (seq dataset subsamples)
    # For simplicity, split flat data for LSTM demo
    train_set = Subset(flat_dataset, train_idx)
    test_set  = Subset(flat_dataset, test_idx)

    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
    test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    print(f'Train: {len(train_set)} windows | Test: {len(test_set)} windows (subject {test_subj})')

In [ ]:
# ── LSTM-only (quick demo) ──
if DATA_LOADED:
    lstm_model = LSTMOnlyModel(
        input_dim=FLAT_DIM,
        n_classes=N_CLASSES,
    )
    print(f'LSTM-only parameters: {sum(p.numel() for p in lstm_model.parameters()):,}')

    # Reshape for LSTM: (batch, seq_len=1, flat_dim)
    class Seq1Dataset(torch.utils.data.Dataset):
        def __init__(self, base_ds):
            self.base = base_ds
        def __len__(self): return len(self.base)
        def __getitem__(self, i):
            x, y = self.base[i]
            return x.unsqueeze(0), y  # add seq dim

    lstm_train = DataLoader(Seq1Dataset(train_set), batch_size=BATCH_SIZE, shuffle=True)
    lstm_test  = DataLoader(Seq1Dataset(test_set),  batch_size=BATCH_SIZE, shuffle=False)

    lstm_result = train_model(
        lstm_model, lstm_train, lstm_test,
        use_adj=False,
        run_name='lstm_only_demo',
    )

## 4. GNN-Only Model

In [ ]:
if DATA_LOADED:
    gnn_train_set = Subset(graph_dataset, train_idx)
    gnn_test_set  = Subset(graph_dataset, test_idx)
    gnn_train_loader = DataLoader(gnn_train_set, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
    gnn_test_loader  = DataLoader(gnn_test_set,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    gnn_model = GNNOnlyModel(node_feat_dim=NODE_FEAT, n_nodes=N_NODES, n_classes=N_CLASSES)
    print(f'GNN-only parameters: {sum(p.numel() for p in gnn_model.parameters()):,}')

    gnn_result = train_model(
        gnn_model, gnn_train_loader, gnn_test_loader,
        use_adj=True,
        run_name='gnn_only_demo',
    )

## 5. GNN + LSTM (Full Model)

In [ ]:
if DATA_LOADED:
    # For GNN+LSTM we need sequence datasets
    # Build separate subject-aware sequence dataset splits
    seq_train_set = Subset(seq_dataset, range(len(seq_dataset)))
    
    # Simple train/test split for demo (proper LOSO in next cell)
    split = int(0.8 * len(seq_dataset))
    seq_train = Subset(seq_dataset, range(split))
    seq_test  = Subset(seq_dataset, range(split, len(seq_dataset)))

    seq_train_loader = DataLoader(seq_train, batch_size=16, shuffle=True,  num_workers=0)
    seq_test_loader  = DataLoader(seq_test,  batch_size=16, shuffle=False, num_workers=0)

    gnn_lstm_model = GNNLSTMModel(
        node_feat_dim=NODE_FEAT,
        n_nodes=N_NODES,
        n_classes=N_CLASSES,
    )
    print(f'GNN+LSTM parameters: {sum(p.numel() for p in gnn_lstm_model.parameters()):,}')

    gnn_lstm_result = train_model(
        gnn_lstm_model, seq_train_loader, seq_test_loader,
        use_adj=True,
        run_name='gnn_lstm_demo',
    )

## 6. Training History Plots

In [ ]:
if DATA_LOADED:
    from src.evaluation import plot_training_history

    for name, result in [
        ('LSTM-only',  lstm_result),
        ('GNN-only',   gnn_result),
        ('GNN+LSTM',   gnn_lstm_result),
    ]:
        print(f'\n── {name} ──')
        plot_training_history(result['history'], run_name=name.lower().replace('+','_').replace('-','_'))

## 7. Model Size Summary

In [ ]:
if DATA_LOADED:
    import pandas as pd
    rows = []
    for name, model in [
        ('LSTM-only',  lstm_model),
        ('GNN-only',   gnn_model),
        ('GNN+LSTM',   gnn_lstm_model),
    ]:
        n_params = sum(p.numel() for p in model.parameters())
        n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        rows.append({'Model': name, 'Total Params': f'{n_params:,}', 'Trainable': f'{n_trainable:,}'})
    pd.DataFrame(rows).set_index('Model')